In [1]:
# mount Drive, load the orders and customers tables, and rebuild the Delivered orders table
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'
so_h = pd.read_csv(BASE_PATH + "sales_orders_header.csv")
customers = pd.read_csv(BASE_PATH + "customers.csv")
print("orders table:", so_h.shape, list(so_h.columns))
print("customers table:", customers.shape, list(customers.columns))

sales_h = so_h[so_h["order_status"] == "Delivered"].copy()
sales_h["order_date"] = pd.to_datetime(sales_h["order_date"])
print("Delivered orders:", len(sales_h))
print("order value of Delivered orders:", sales_h["total_order_value"].sum())
print("customers in the customers table:", customers["customer_id"].nunique())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
orders table: (20000, 11) ['so_id', 'customer_id', 'branch_id', 'order_date', 'delivery_date', 'order_status', 'payment_terms', 'total_order_value', 'total_gst_amount', 'grand_total', 'sales_channel']
customers table: (500, 15) ['customer_id', 'customer_type', 'industry_segment', 'city', 'state', 'pincode', 'region', 'branch_id', 'credit_limit', 'current_balance', 'payment_terms', 'customer_since', 'last_purchase_date', 'total_purchase_value', 'customer_rating']
Delivered orders: 18033
order value of Delivered orders: 22958987430
customers in the customers table: 500


In [2]:
cust_value = (sales_h.groupby('customer_id')
              .agg(orders=('so_id', 'count'), money=('total_order_value', 'sum'))
              .reset_index()
              .sort_values('money', ascending=False)
              .reset_index(drop=True))
cust_value['rank'] = cust_value.index + 1

print("customers ranked:", len(cust_value))
print("money added over customers:", cust_value['money'].sum())
print("money in sales_h:", sales_h['total_order_value'].sum())
print("difference:", cust_value['money'].sum() - sales_h['total_order_value'].sum())
print("smallest customer:", cust_value['money'].min())
print("biggest customer:", cust_value['money'].max())
print("average per customer:", round(cust_value['money'].mean()))

top20 = cust_value.head(20).merge(
    customers[['customer_id', 'customer_type', 'branch_id']],
    on='customer_id', how='left')
print(top20[['rank', 'customer_id', 'customer_type', 'branch_id', 'orders', 'money']].to_string(index=False))

customers ranked: 500
money added over customers: 22958987430
money in sales_h: 22958987430
difference: 0
smallest customer: 17721530
biggest customer: 84966970
average per customer: 45917975
 rank customer_id customer_type branch_id  orders    money
    1       C0153        Retail    PUN001      56 84966970
    2       C0288        Dealer    DEL001      55 76291160
    3       C0483     Corporate    DEL001      47 75352940
    4       C0475   Fleet Owner    KOL001      55 74442140
    5       C0495   Fleet Owner    AHM001      48 74165320
    6       C0240     Corporate    HYD001      46 72778580
    7       C0420        Retail    CHN001      46 72545910
    8       C0110        Dealer    CHN001      42 71230450
    9       C0083        Dealer    DEL001      45 70378740
   10       C0206        Retail    PUN001      40 70279540
   11       C0178        Dealer    KOL001      54 70236970
   12       C0254        Dealer    AHM001      48 70227620
   13       C0466        Retail    DEL001

In [3]:
total_money = cust_value['money'].sum()

for n in [10, 50, 100]:
    top_money = cust_value.head(n)['money'].sum()
    share = top_money / total_money * 100
    even_share = n / len(cust_value) * 100
    print("top", n, "customers hold", round(share, 2), "% of the money; an even spread would give", round(even_share, 2), "%")

top 10 customers hold 3.23 % of the money; an even spread would give 2.0 %
top 50 customers hold 14.33 % of the money; an even spread would give 10.0 %
top 100 customers hold 26.63 % of the money; an even spread would give 20.0 %


In [4]:
type_view = cust_value.merge(customers[['customer_id', 'customer_type']], on='customer_id', how='left')

print("customers with no type:", type_view['customer_type'].isna().sum())

by_type = (type_view.groupby('customer_type')
           .agg(customers=('customer_id', 'count'), orders=('orders', 'sum'), money=('money', 'sum'))
           .sort_values('money', ascending=False))
by_type['average_per_customer'] = (by_type['money'] / by_type['customers']).round()
by_type['share_of_money_%'] = (by_type['money'] / by_type['money'].sum() * 100).round(2)

print(by_type)
print("customers added over types:", by_type['customers'].sum())
print("money added over types:", by_type['money'].sum())
print("difference to total:", by_type['money'].sum() - sales_h['total_order_value'].sum())

customers with no type: 0
               customers  orders       money  average_per_customer  \
customer_type                                                        
Government           114    4075  5049342920            44292482.0   
Corporate            104    3672  4717364970            45359279.0   
Dealer                99    3633  4683087090            47303910.0   
Fleet Owner           95    3457  4411907790            46441135.0   
Retail                88    3196  4097284660            46560053.0   

               share_of_money_%  
customer_type                    
Government                21.99  
Corporate                 20.55  
Dealer                    20.40  
Fleet Owner               19.22  
Retail                    17.85  
customers added over types: 500
money added over types: 22958987430
difference to total: 0


In [5]:
branch_view = cust_value.merge(customers[['customer_id', 'branch_id']], on='customer_id', how='left')

print("customers with no branch:", branch_view['branch_id'].isna().sum())

by_branch = (branch_view.groupby('branch_id')
             .agg(customers=('customer_id', 'count'), orders=('orders', 'sum'), money=('money', 'sum'))
             .sort_values('money', ascending=False))
by_branch['average_per_customer'] = (by_branch['money'] / by_branch['customers']).round()
by_branch['share_of_money_%'] = (by_branch['money'] / by_branch['money'].sum() * 100).round(2)

print(by_branch)
print("customers added over branches:", by_branch['customers'].sum())
print("money added over branches:", by_branch['money'].sum())
print("difference to total:", by_branch['money'].sum() - sales_h['total_order_value'].sum())

customers with no branch: 0
           customers  orders       money  average_per_customer  \
branch_id                                                        
CHN001           102    3677  4682202010            45903941.0   
KOL001            96    3413  4339861320            45206889.0   
AHM001            88    3115  3986314570            45299029.0   
DEL001            71    2682  3500031300            49296215.0   
PUN001            75    2668  3373530010            44980400.0   
HYD001            68    2478  3077048220            45250709.0   

           share_of_money_%  
branch_id                    
CHN001                20.39  
KOL001                18.90  
AHM001                17.36  
DEL001                15.24  
PUN001                14.69  
HYD001                13.40  
customers added over branches: 500
money added over branches: 22958987430
difference to total: 0


In [6]:
region_view = cust_value.merge(customers[['customer_id', 'region']], on='customer_id', how='left')

print("customers with no region:", region_view['region'].isna().sum())

by_region = (region_view.groupby('region')
             .agg(customers=('customer_id', 'count'), orders=('orders', 'sum'), money=('money', 'sum'))
             .sort_values('money', ascending=False))
by_region['average_per_customer'] = (by_region['money'] / by_region['customers']).round()
by_region['share_of_money_%'] = (by_region['money'] / by_region['money'].sum() * 100).round(2)

print(by_region)
print("customers added over regions:", by_region['customers'].sum())
print("money added over regions:", by_region['money'].sum())
print("difference to total:", by_region['money'].sum() - sales_h['total_order_value'].sum())

customers with no region: 0
            customers  orders       money  average_per_customer  \
region                                                            
South             158    5722  7261744460            45960408.0   
West              151    5335  6730779040            44574696.0   
East               87    3161  4051950040            46574138.0   
North              82    3029  3938165590            48026410.0   
Central            11     406   501842570            45622052.0   
North East         11     380   474505730            43136885.0   

            share_of_money_%  
region                        
South                  31.63  
West                   29.32  
East                   17.65  
North                  17.15  
Central                 2.19  
North East              2.07  
customers added over regions: 500
money added over regions: 22958987430
difference to total: 0


In [7]:
industry_view = cust_value.merge(customers[['customer_id', 'industry_segment']], on='customer_id', how='left')

print("customers with no industry segment:", industry_view['industry_segment'].isna().sum())

by_industry = (industry_view.groupby('industry_segment')
               .agg(customers=('customer_id', 'count'), orders=('orders', 'sum'), money=('money', 'sum'))
               .sort_values('money', ascending=False))
by_industry['average_per_customer'] = (by_industry['money'] / by_industry['customers']).round()
by_industry['share_of_money_%'] = (by_industry['money'] / by_industry['money'].sum() * 100).round(2)

print(by_industry.to_string())
print("customers added over industry segments:", by_industry['customers'].sum())
print("money added over industry segments:", by_industry['money'].sum())
print("difference to total:", by_industry['money'].sum() - sales_h['total_order_value'].sum())

customers with no industry segment: 0
                  customers  orders       money  average_per_customer  share_of_money_%
industry_segment                                                                       
Logistics                96    3461  4392333800            45753477.0             19.13
Mining                   98    3394  4249857340            43365891.0             18.51
Construction             90    3214  4028671970            44763022.0             17.55
Infrastructure           79    2864  3695216850            46774897.0             16.09
Manufacturing            70    2649  3401175350            48588219.0             14.81
Rental                   67    2451  3191732120            47637793.0             13.90
customers added over industry segments: 500
money added over industry segments: 22958987430
difference to total: 0


In [8]:
sales_year = sales_h.copy()
sales_year['year'] = pd.to_datetime(sales_year['order_date']).dt.year

print("orders with no date:", sales_year['order_date'].isna().sum())
print("first order date:", pd.to_datetime(sales_year['order_date']).min())
print("last order date:", pd.to_datetime(sales_year['order_date']).max())

by_year = (sales_year.groupby('year')
           .agg(orders=('so_id', 'count'), customers_buying=('customer_id', 'nunique'), money=('total_order_value', 'sum')))
by_year['average_order_value'] = (by_year['money'] / by_year['orders']).round()
by_year['share_of_money_%'] = (by_year['money'] / by_year['money'].sum() * 100).round(2)
by_year['change_vs_previous_year_%'] = (by_year['money'].pct_change() * 100).round(2)

print(by_year.to_string())
print("orders added over years:", by_year['orders'].sum())
print("money added over years:", by_year['money'].sum())
print("difference to total:", by_year['money'].sum() - sales_h['total_order_value'].sum())

orders with no date: 0
first order date: 2019-01-01 00:00:00
last order date: 2024-12-31 00:00:00
      orders  customers_buying       money  average_order_value  share_of_money_%  change_vs_previous_year_%
year                                                                                                        
2019    3031               500  3870016600            1276812.0             16.86                        NaN
2020    3038               500  3917532120            1289510.0             17.06                       1.23
2021    3038               498  3852479620            1268097.0             16.78                      -1.66
2022    2948               498  3816288220            1294535.0             16.62                      -0.94
2023    3022               500  3824675310            1265611.0             16.66                       0.22
2024    2956               500  3677995560            1244247.0             16.02                      -3.84
orders added over years: 18033

In [9]:
cust_value['average_order_value'] = (cust_value['money'] / cust_value['orders']).round()

summary = cust_value[['orders', 'average_order_value', 'money']].describe(percentiles=[0.25, 0.5, 0.75]).round()
print(summary.to_string())

print("customers with the biggest average order:")
print(cust_value.sort_values('average_order_value', ascending=False).head(5)[['customer_id', 'orders', 'average_order_value', 'money']].to_string(index=False))
print("customers with the smallest average order:")
print(cust_value.sort_values('average_order_value').head(5)[['customer_id', 'orders', 'average_order_value', 'money']].to_string(index=False))

       orders  average_order_value       money
count   500.0                500.0       500.0
mean     36.0            1273690.0  45917975.0
std       6.0             197849.0  10298204.0
min      21.0             738397.0  17721530.0
25%      32.0            1131625.0  38446610.0
50%      36.0            1269734.0  45321980.0
75%      40.0            1395526.0  52438192.0
max      56.0            1955791.0  84966970.0
customers with the biggest average order:
customer_id  orders  average_order_value    money
      C0234      29            1955791.0 56717930
      C0491      27            1894105.0 51140830
      C0482      31            1866024.0 57846740
      C0498      28            1815909.0 50845460
      C0191      29            1778374.0 51572860
customers with the smallest average order:
customer_id  orders  average_order_value    money
      C0347      24             738397.0 17721530
      C0174      29             806422.0 23386250
      C0145      33             820578.0 2

In [10]:
sales_columns = ['so_id', 'customer_id', 'order_date', 'total_order_value']
customer_columns = ['customer_id', 'customer_type', 'branch_id', 'region', 'industry_segment']

print("empty cells in the delivered orders table:")
print(sales_h[sales_columns].isnull().sum())

print("empty cells in the customers table:")
print(customers[customer_columns].isnull().sum())

print("blank text in the customers columns:")
print(customers[customer_columns].astype(str).apply(lambda column: column.str.strip() == '').sum())

print("orders with a value of zero or less:", (sales_h['total_order_value'] <= 0).sum())
print("customers with money of zero or less:", (cust_value['money'] <= 0).sum())
print("customers with zero orders:", (cust_value['orders'] <= 0).sum())

empty cells in the delivered orders table:
so_id                0
customer_id          0
order_date           0
total_order_value    0
dtype: int64
empty cells in the customers table:
customer_id         0
customer_type       0
branch_id           0
region              0
industry_segment    0
dtype: int64
blank text in the customers columns:
customer_id         0
customer_type       0
branch_id           0
region              0
industry_segment    0
dtype: int64
orders with a value of zero or less: 0
customers with money of zero or less: 0
customers with zero orders: 0


In [11]:
print("orders whose customer is not in the customers table:", (~sales_h['customer_id'].isin(customers['customer_id'])).sum())
print("customers with no delivered order:", (~customers['customer_id'].isin(sales_h['customer_id'])).sum())

order_years = pd.to_datetime(sales_h['order_date']).dt.year
orders_by_customer_year = (sales_h.assign(year=order_years)
                           .groupby(['customer_id', 'year']).size()
                           .unstack(fill_value=0)
                           .reindex(customers['customer_id'], fill_value=0))

print("customer and year cells with no order:", (orders_by_customer_year == 0).sum().sum(), "out of", orders_by_customer_year.size)
print("customers who had a year with no order:")
print(orders_by_customer_year[(orders_by_customer_year == 0).any(axis=1)].to_string())

orders whose customer is not in the customers table: 0
customers with no delivered order: 0
customer and year cells with no order: 4 out of 3000
customers who had a year with no order:
year         2019  2020  2021  2022  2023  2024
customer_id                                    
C0133           3     5     0     9     9     3
C0143           7     5     0     5     5     5
C0344           6     8     4     0     3     7
C0482           1     6     8     0     7     9


In [12]:
print("rows in the delivered orders table:", len(sales_h))
print("unique order IDs in the delivered orders table:", sales_h['so_id'].nunique())
print("repeated order IDs:", sales_h['so_id'].duplicated().sum())
print("repeated customer IDs in the customers table:", customers['customer_id'].duplicated().sum())
print("repeated customer IDs in the ranking:", cust_value['customer_id'].duplicated().sum())
print("orders sharing the same customer, date and value with another order:",
      sales_h.duplicated(subset=['customer_id', 'order_date', 'total_order_value'], keep=False).sum())

rows in the delivered orders table: 18033
unique order IDs in the delivered orders table: 18033
repeated order IDs: 0
repeated customer IDs in the customers table: 0
repeated customer IDs in the ranking: 0
orders sharing the same customer, date and value with another order: 0


In [13]:
delivered_from_file = so_h[so_h['order_status'] == 'Delivered']
total_money = sales_h['total_order_value'].sum()

print("orders by status in the orders file:")
print(so_h['order_status'].value_counts())
print("delivered orders straight from the orders file:", len(delivered_from_file))
print("delivered money straight from the orders file:", delivered_from_file['total_order_value'].sum())
print("delivered orders used in this KPI:", len(sales_h))
print("money used in this KPI:", total_money)
print("difference to the orders file:", total_money - delivered_from_file['total_order_value'].sum())

cut_totals = {
    'customer ranking': cust_value['money'].sum(),
    'customer type': by_type['money'].sum(),
    'branch': by_branch['money'].sum(),
    'region': by_region['money'].sum(),
    'industry segment': by_industry['money'].sum(),
    'year': by_year['money'].sum(),
}
for cut_name, cut_money in cut_totals.items():
    print(cut_name, "adds to", cut_money, "| difference to total:", cut_money - total_money)

orders by status in the orders file:
order_status
Delivered    18033
Cancelled     1967
Name: count, dtype: int64
delivered orders straight from the orders file: 18033
delivered money straight from the orders file: 22958987430
delivered orders used in this KPI: 18033
money used in this KPI: 22958987430
difference to the orders file: 0
customer ranking adds to 22958987430 | difference to total: 0
customer type adds to 22958987430 | difference to total: 0
branch adds to 22958987430 | difference to total: 0
region adds to 22958987430 | difference to total: 0
industry segment adds to 22958987430 | difference to total: 0
year adds to 22958987430 | difference to total: 0


In [14]:
import numpy as np

money_pivot = pd.pivot_table(sales_h, index='customer_id', values='total_order_value', aggfunc='sum')['total_order_value'].sort_values(ascending=False)
print("customers in the pivot table:", len(money_pivot))
print("largest difference in money per customer:", np.abs(money_pivot.values - cust_value['money'].values).max())
print("same customer in first place:", money_pivot.index[0] == cust_value['customer_id'].iloc[0])

customer_details = customers[['customer_id', 'customer_type', 'branch_id', 'region', 'industry_segment']].rename(columns={'branch_id': 'customer_branch'})
orders_with_customer = sales_h.merge(customer_details, on='customer_id', how='left')

print("orders whose own branch differs from the customer's branch:", (orders_with_customer['branch_id'] != orders_with_customer['customer_branch']).sum())

for column_name, first_way in [('customer_type', by_type), ('customer_branch', by_branch), ('region', by_region), ('industry_segment', by_industry)]:
    second_way = orders_with_customer.groupby(column_name)['total_order_value'].sum()
    print(column_name, "largest difference between the two ways:", (second_way - first_way['money'].reindex(second_way.index)).abs().max())

order_dates = pd.to_datetime(sales_h['order_date'])
for year in range(2019, 2025):
    money_in_year = sales_h.loc[(order_dates >= f'{year}-01-01') & (order_dates <= f'{year}-12-31'), 'total_order_value'].sum()
    print(year, "money by date range:", money_in_year, "| difference to the year table:", money_in_year - by_year.loc[year, 'money'])

for n in [10, 50, 100]:
    print("top", n, "customers hold", round(money_pivot.nlargest(n).sum() / money_pivot.sum() * 100, 2), "% of the money")

print("overall average order value:", round(sales_h['total_order_value'].sum() / len(sales_h)))

customers in the pivot table: 500
largest difference in money per customer: 0
same customer in first place: True
orders whose own branch differs from the customer's branch: 0
customer_type largest difference between the two ways: 0
customer_branch largest difference between the two ways: 0
region largest difference between the two ways: 0
industry_segment largest difference between the two ways: 0
2019 money by date range: 3870016600 | difference to the year table: 0
2020 money by date range: 3917532120 | difference to the year table: 0
2021 money by date range: 3852479620 | difference to the year table: 0
2022 money by date range: 3816288220 | difference to the year table: 0
2023 money by date range: 3824675310 | difference to the year table: 0
2024 money by date range: 3677995560 | difference to the year table: 0
top 10 customers hold 3.23 % of the money
top 50 customers hold 14.33 % of the money
top 100 customers hold 26.63 % of the money
overall average order value: 1273165


In [15]:
cust_check = cust_value.set_index('customer_id').copy()
cust_check['largest_order'] = sales_h.groupby('customer_id')['total_order_value'].max()
cust_check['largest_order_vs_average'] = cust_check['largest_order'] / cust_check['average_order_value']
cust_check['largest_order_share_%'] = cust_check['largest_order'] / cust_check['money'] * 100

print("largest single order in the whole table:", sales_h['total_order_value'].max())
print("largest order compared with the customer's average order: middle", round(cust_check['largest_order_vs_average'].median(), 2), "| highest", round(cust_check['largest_order_vs_average'].max(), 2), "for customer", cust_check['largest_order_vs_average'].idxmax())
print("share of a customer's money from their largest order: middle", round(cust_check['largest_order_share_%'].median(), 2), "% | highest", round(cust_check['largest_order_share_%'].max(), 2), "% for customer", cust_check['largest_order_share_%'].idxmax())

top20_ids = cust_value.head(20)['customer_id']
print("top 20 customers, highest share of money from one order:", round(cust_check.loc[top20_ids, 'largest_order_share_%'].max(), 2), "%")

money_without_largest = (cust_check['money'] - cust_check['largest_order']).sort_values(ascending=False)
print("top 20 customers still in the top 20 after removing each customer's largest order:", len(set(money_without_largest.head(20).index) & set(top20_ids)), "out of 20")

largest single order in the whole table: 8169760
largest order compared with the customer's average order: middle 3.23 | highest 5.86 for customer C0225
share of a customer's money from their largest order: middle 9.08 % | highest 17.38 % for customer C0349
top 20 customers, highest share of money from one order: 11.53 %
top 20 customers still in the top 20 after removing each customer's largest order: 19 out of 20


In [16]:
dropped_out = set(top20_ids) - set(money_without_largest.head(20).index)
came_in = set(money_without_largest.head(20).index) - set(top20_ids)
rank_without_largest = pd.Series(range(1, len(money_without_largest) + 1), index=money_without_largest.index)

for customer in dropped_out:
    print("dropped out:", customer, "| original rank:", cust_check.loc[customer, 'rank'], "| largest order:", cust_check.loc[customer, 'largest_order'],
          "| share from largest order:", round(cust_check.loc[customer, 'largest_order_share_%'], 2), "% | rank without it:", rank_without_largest[customer])
for customer in came_in:
    print("came in:", customer, "| original rank:", cust_check.loc[customer, 'rank'], "| largest order:", cust_check.loc[customer, 'largest_order'],
          "| share from largest order:", round(cust_check.loc[customer, 'largest_order_share_%'], 2), "% | rank without it:", rank_without_largest[customer])

dropped out: C0359 | original rank: 17 | largest order: 7752220 | share from largest order: 11.53 % | rank without it: 26
came in: C0490 | original rank: 21 | largest order: 4257600 | share from largest order: 6.4 % | rank without it: 20


In [17]:
order_dates = pd.to_datetime(sales_h['order_date'])
order_month = order_dates.dt.to_period('M')

orders_per_month = sales_h.groupby(order_month).size()
days_with_orders_per_month = order_dates.groupby(order_month).nunique()

print("months with orders:", len(orders_per_month), "of 72")
print("fewest orders in a month:", orders_per_month.min(), "in", orders_per_month.idxmin(), "| most:", orders_per_month.max(), "in", orders_per_month.idxmax(), "| average:", round(orders_per_month.mean(), 1))
print("orders in the first month:", orders_per_month.iloc[0], "| orders in the last month:", orders_per_month.iloc[-1])
print("days with orders in the first month:", days_with_orders_per_month.iloc[0], "| in the last month:", days_with_orders_per_month.iloc[-1], "| middle month:", days_with_orders_per_month.median())

first_and_last_order = sales_h.assign(order_day=order_dates).groupby('customer_id')['order_day'].agg(['min', 'max'])
print("earliest first order among customers:", first_and_last_order['min'].min(), "| latest first order among customers:", first_and_last_order['min'].max())
print("earliest last order among customers:", first_and_last_order['max'].min(), "| latest last order among customers:", first_and_last_order['max'].max())
print("customers whose first order is after 30 Jun 2019:", (first_and_last_order['min'] > '2019-06-30').sum())
print("customers whose last order is before 1 Jul 2024:", (first_and_last_order['max'] < '2024-07-01').sum())

months with orders: 72 of 72
fewest orders in a month: 209 in 2024-05 | most: 312 in 2020-07 | average: 250.5
orders in the first month: 254 | orders in the last month: 223
days with orders in the first month: 31 | in the last month: 31 | middle month: 31.0
earliest first order among customers: 2019-01-01 00:00:00 | latest first order among customers: 2019-12-18 00:00:00
earliest last order among customers: 2024-01-08 00:00:00 | latest last order among customers: 2024-12-31 00:00:00
customers whose first order is after 30 Jun 2019: 31
customers whose last order is before 1 Jul 2024: 23


In [18]:
customer_history = cust_value.set_index('customer_id').join(first_and_last_order)
late_start = customer_history['min'] > '2019-06-30'
early_end = customer_history['max'] < '2024-07-01'
bottom_50_ids = set(cust_value.tail(50)['customer_id'])

for label, group_mask in [('first order after 30 Jun 2019', late_start), ('last order before 1 Jul 2024', early_end), ('neither', ~(late_start | early_end))]:
    group = customer_history[group_mask]
    print(label, "| customers:", len(group), "| average orders:", round(group['orders'].mean(), 1), "| average money:", round(group['money'].mean()),
          "| average order value:", round(group['average_order_value'].mean()), "| in the bottom 50:", len(set(group.index) & bottom_50_ids))

first order after 30 Jun 2019 | customers: 31 | average orders: 35.1 | average money: 46313703 | average order value: 1327312 | in the bottom 50: 3
last order before 1 Jul 2024 | customers: 23 | average orders: 31.9 | average money: 41017466 | average order value: 1284819 | in the bottom 50: 4
neither | customers: 448 | average orders: 36.3 | average money: 46118212 | average order value: 1270065 | in the bottom 50: 43


In [19]:
middle_customer = cust_value.iloc[249]['customer_id']

for customer in ['C0153', 'C0347', middle_customer]:
    raw_rows = so_h[(so_h['customer_id'] == customer) & (so_h['order_status'] == 'Delivered')]
    cancelled_rows = so_h[(so_h['customer_id'] == customer) & (so_h['order_status'] == 'Cancelled')]

    raw_money = 0
    for order_value in raw_rows['total_order_value']:
        raw_money = raw_money + order_value

    table_row = cust_value[cust_value['customer_id'] == customer].iloc[0]
    customer_row = customers[customers['customer_id'] == customer].iloc[0]
    raw_dates = pd.to_datetime(raw_rows['order_date'])

    print(customer, "| rank in the table:", table_row['rank'])
    print("  orders: raw file", len(raw_rows), "| table", table_row['orders'])
    print("  money: raw file", raw_money, "| table", table_row['money'])
    print("  largest order: raw file", raw_rows['total_order_value'].max(), "| table", cust_check.loc[customer, 'largest_order'])
    print("  first and last order: raw file", raw_dates.min().date(), raw_dates.max().date(), "| table", first_and_last_order.loc[customer, 'min'].date(), first_and_last_order.loc[customer, 'max'].date())
    print("  type and branch in the customers file:", customer_row['customer_type'], customer_row['branch_id'])
    print("  cancelled orders left out:", len(cancelled_rows), "worth", cancelled_rows['total_order_value'].sum())

C0153 | rank in the table: 1
  orders: raw file 56 | table 56
  money: raw file 84966970 | table 84966970
  largest order: raw file 6716250 | table 6716250
  first and last order: raw file 2019-01-17 2024-12-12 | table 2019-01-17 2024-12-12
  type and branch in the customers file: Retail PUN001
  cancelled orders left out: 3 worth 910050
C0347 | rank in the table: 500
  orders: raw file 24 | table 24
  money: raw file 17721530 | table 17721530
  largest order: raw file 1928300 | table 1928300
  first and last order: raw file 2019-03-04 2024-12-03 | table 2019-03-04 2024-12-03
  type and branch in the customers file: Retail DEL001
  cancelled orders left out: 3 worth 5130870
C0436 | rank in the table: 250
  orders: raw file 34 | table 34
  money: raw file 45357190 | table 45357190
  largest order: raw file 2984530 | table 2984530
  first and last order: raw file 2019-02-01 2024-12-15 | table 2019-02-01 2024-12-15
  type and branch in the customers file: Fleet Owner KOL001
  cancelled or